# Load Fix Verification

Verifies that the three load signals are now **consistent** after the fix:

| Signal | Before fix | After fix |
|---|---|---|
| Observation slot `[27:30]` | `upper_demand_prbs / n_prbs` | `gnb_total_useful_load` |
| Jain fairness (reward) | demand PRBs | window-avg useful PRBs |
| Info dict | no per-gNB useful load vector | `gnb_total_useful_load_end` + StdDev |

The canonical load is now:
```
gnb_total_useful_load[g] = Σ_s  window_avg_useful_prbs[g,s] / n_prbs[g]   ∈ [0,1]
```
This is identical to the KPM PRB load % in the paper.

In [ ]:
import os, sys
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-chech')
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scenario_creator import create_multignb_env
from global_ppo_3gnb_env import GlobalPPO3GNBEnv

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
SLICE_TYPES = ('eMBB', 'URLLC', 'mMTC')
print('Ready')

---
## 1 — Observation shape: slot `[27:30]` must equal `Σ_s loads[g,s]`

In [ ]:
env = GlobalPPO3GNBEnv(
    seed=42,
    scenario_mode='snapshot',
    snapshot_scenario='embb_g0_offload',
    warmup_steps=2,
    safe_admission_enabled=False,
    terminal_reward_only=False,
    global_steps_per_episode=10,
)

obs, info = env.reset()

N_GNBS   = env.n_gnbs           # 3
N_SLICES = len(env.slice_types) # 3
LOAD_SLOT_START = N_GNBS * N_SLICES * 3  # = 27

obs_loads      = obs[0 : N_GNBS * N_SLICES].reshape(N_GNBS, N_SLICES)
obs_gnb_useful = obs[LOAD_SLOT_START : LOAD_SLOT_START + N_GNBS]
computed       = np.clip(obs_loads.sum(axis=1), 0, 1)

print('obs shape:', obs.shape, '  (expected 48)')
print()
print('obs_loads per-gNB per-slice (window-avg useful):')
print(obs_loads.round(4))
print()
print('obs slot [27:30] gnb_useful_load  :', obs_gnb_useful.round(4))
print('Σ_slices obs_loads per gNB        :', computed.round(4))
print()
ok = np.allclose(obs_gnb_useful, computed, atol=1e-5)
print('Match:', '✓ YES' if ok else '✗ NO')

---
## 2 — Info dict: `gnb_total_useful_load_end` and Jain signals

In [ ]:
zero = np.zeros(env.action_space.shape, dtype=np.float32)
records = []

for step in range(10):
    obs, rew, term, trunc, info = env.step(zero)

    useful_mat   = info['useful_load_matrix_end']     # shape (3,3)
    gnb_load_vec = info['gnb_total_useful_load_end']  # shape (3,)
    computed_vec = np.clip(useful_mat.sum(axis=1), 0, 1)

    demand_per_gnb  = info['jain_per_gnb_demand_prbs']
    demand_load_vec = demand_per_gnb / 100.0

    records.append({
        'step'           : step + 1,
        'gnb0_useful'    : float(gnb_load_vec[0]),
        'gnb1_useful'    : float(gnb_load_vec[1]),
        'gnb2_useful'    : float(gnb_load_vec[2]),
        'std_useful'     : float(info['gnb_total_useful_load_std_end']),
        'gnb0_demand'    : float(demand_load_vec[0]),
        'gnb1_demand'    : float(demand_load_vec[1]),
        'gnb2_demand'    : float(demand_load_vec[2]),
        'jain_useful'    : float(info['jain_fairness_raw']),
        'jain_demand'    : float(info['jain_demand_raw']),
        'vec_ok'         : np.allclose(gnb_load_vec, computed_vec, atol=1e-5),
    })
    if trunc:
        break

env.close()
df = pd.DataFrame(records)

print(df[[
    'step',
    'gnb0_useful','gnb1_useful','gnb2_useful','std_useful',
    'gnb0_demand','gnb1_demand','gnb2_demand',
    'jain_useful','jain_demand','vec_ok',
]].round(4).to_string(index=False))

print()
print('All vec_ok:', '✓ YES' if df.vec_ok.all() else '✗ NO')

---
## 3 — Visual: useful load vs demand load

In [ ]:
gnb_colors = ['steelblue', 'tomato', 'seagreen']
steps = df.step.values

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
ax_load, ax_jain = axes

for g, color in enumerate(gnb_colors):
    ax_load.plot(steps, df[f'gnb{g}_useful'], 'o-', color=color, lw=2,
                 label=f'gNB-{g} useful (obs + reward)')
    ax_load.axhline(df[f'gnb{g}_demand'].iloc[0], color=color, lw=1.2,
                    linestyle='--', alpha=0.5,
                    label=f'gNB-{g} demand (fixed, diagnostic only)')

ax_load.set_ylabel('Per-gNB total PRB load  [0,1]')
ax_load.set_title(
    'Useful load (solid) vs demand load (dashed)\n'
    'Obs slot [27:30] and Jain reward now use USEFUL — demand is diagnostic only'
)
ax_load.legend(fontsize=8, ncol=2)
ax_load.grid(alpha=0.4)

ax_jain.plot(steps, df['jain_useful'], 'o-', color='purple', lw=2,
             label='Jain (useful PRBs) — used in reward')
ax_jain.plot(steps, df['jain_demand'], 's--', color='gray', lw=1.5,
             label='Jain (demand PRBs) — diagnostic only')
ax_jain.set_ylabel('Jain fairness')
ax_jain.set_xlabel('Upper step')
ax_jain.set_ylim(0.5, 1.05)
ax_jain.legend(fontsize=9)
ax_jain.grid(alpha=0.4)

plt.tight_layout()
plt.show()

---
## 4 — Handover test: 3 UEs, low demand, visible migration

**Setup:** 3 eMBB UEs placed close to gNB-0 (~15 PRBs each = 45% total load).  
gNB-1 and gNB-2 start empty.  
- t=1s → UE-0 moved to gNB-1  
- t=3s → UE-1 moved to gNB-2  

**Expected:** gNB-0 drops 45% → 30% → 15%.  StdDev drops each step — the paper's reward improving.

In [ ]:
TICK_S       = 0.001
SAMPLE_EVERY = 10       # record every 10 ms
SIMULATION_S = 6.0
N_TICKS      = int(SIMULATION_S / TICK_S)

GNB_CONFIGS = [
    {'id': 0, 'x':   0.0, 'y':   0.0, 'coverage_radius': 520.0, 'carrier_id': 0, 'n_prbs': 100},
    {'id': 1, 'x': 450.0, 'y':   0.0, 'coverage_radius': 520.0, 'carrier_id': 0, 'n_prbs': 100},
    {'id': 2, 'x': 225.0, 'y': 390.0, 'coverage_radius': 520.0, 'carrier_id': 0, 'n_prbs': 100},
]

# UE positions: start near gNB-0, re-placed near target after HO
UE_POS_G0 = [(25,  10), (10, -20), (30,  20)]
UE_POS_G1 = [(435, 10), (425,-20), (440, 20)]
UE_POS_G2 = [(235,380), (220, 400),(245,375)]

# (at_tick, ue_index, target_gnb, new_position)
TRANSFERS = [
    (int(1.0 / TICK_S), 0, 1, UE_POS_G1[0]),
    (int(3.0 / TICK_S), 1, 2, UE_POS_G2[1]),
]
transfer_ticks = {t[0] for t in TRANSFERS}
transfer_queue = list(TRANSFERS)

base_env = create_multignb_env(
    rng=np.random.default_rng(0),
    n=4,
    gnb_configs=GNB_CONFIGS,
    slots_per_step=1,
    L1_level=False,
    step_dt=TICK_S,
    mobility_dt=TICK_S,
    radio_substeps=1,
    max_episode_steps=N_TICKS + 10,
    max_handovers_per_ue_episode=0,   # manual HOs only
)

# 3 eMBB UEs near gNB-0, each ~15 PRBs at high SINR
# 12 Mbps × 1 ms = 12 000 bits/tick; at ~800 bits/PRB → 15 PRBs
all_ues = []
for x, y in UE_POS_G0:
    uid = base_env.add_ue(
        x=float(x), y=float(y), vx=0., vy=0.,
        slice_type='eMBB',
        bit_rate=12e6,
        packet_size_bits=12_000.,
        traffic_model='fixed_packet_cbr',
    )
    all_ues.append(base_env.get_ue(uid))

print('Initial assignments:')
for ue in all_ues:
    print(f'  UE-{ue.id}  pos=({ue.x:.0f},{ue.y:.0f})  gNB={ue.serving_gnb}')

# Per-gNB total useful load — the paper's KPM metric
def gnb_total_useful_load(env):
    loads = env.get_slice_loads()   # {(gnb_id, slice): useful_prbs/n_prbs}
    return np.clip(
        np.array([
            sum(loads.get((g, s), 0.0) for s in SLICE_TYPES)
            for g in range(3)
        ]),
        0.0, 1.0,
    )

In [ ]:
transfer_log = []
records_ho   = []

for tick in range(N_TICKS):

    # Manual transfer
    if tick in transfer_ticks and transfer_queue:
        _, ue_idx, tgt_gnb, (nx, ny) = transfer_queue.pop(0)
        ue  = all_ues[ue_idx]
        src = int(ue.serving_gnb)
        ue.x, ue.y = float(nx), float(ny)
        base_env._perform_handover(
            ue,
            base_env._gnb_by_id[src],
            base_env._gnb_by_id[tgt_gnb],
        )
        ue.queue = 0.0
        t_s = tick * TICK_S
        transfer_log.append((t_s, ue.id, src, tgt_gnb))
        print(f't={t_s:.1f}s  UE-{ue.id}: gNB-{src} → gNB-{tgt_gnb}')

    base_env.step(0)

    if tick % SAMPLE_EVERY == 0:
        gl = gnb_total_useful_load(base_env)
        records_ho.append({
            'time_s': (tick + 1) * TICK_S,
            'gnb0'  : float(gl[0]),
            'gnb1'  : float(gl[1]),
            'gnb2'  : float(gl[2]),
            'std'   : float(np.std(gl)),
        })

base_env.close()
df_ho = pd.DataFrame(records_ho)
print(f'\nCollected {len(df_ho)} samples')

print('\nLoad at key moments:')
for t_check, label in [
    (0.5, 'before any HO'),
    (1.5, 'after UE-0 → gNB-1'),
    (4.0, 'after UE-1 → gNB-2'),
]:
    row = df_ho.iloc[(df_ho.time_s - t_check).abs().argmin()]
    print(f'  t={t_check:.1f}s ({label}):  '
          f'gNB-0={row.gnb0:.3f}  gNB-1={row.gnb1:.3f}  '
          f'gNB-2={row.gnb2:.3f}  std={row["std"]:.4f}')

In [ ]:
gnb_colors = ['steelblue', 'tomato', 'seagreen']
gnb_labels = ['gNB-0 (source)', 'gNB-1 (gets UE-0 at t=1s)', 'gNB-2 (gets UE-1 at t=3s)']

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})
ax_load, ax_std = axes

for g, (color, label) in enumerate(zip(gnb_colors, gnb_labels)):
    ax_load.plot(df_ho.time_s, df_ho[f'gnb{g}'],
                 color=color, lw=2.5, label=label)

for t_s, uid, src, tgt in transfer_log:
    ax_load.axvline(t_s, color='crimson', lw=2, linestyle='--', alpha=0.8)
    ax_load.annotate(f'UE-{uid}\n{src}→{tgt}',
                     xy=(t_s + 0.05, 0.50), fontsize=9, color='crimson')

ax_load.set_ylabel('Per-gNB total useful PRB load [0,1]\n= Σ_slices useful_prbs / n_prbs')
ax_load.set_ylim(-0.02, 0.70)
ax_load.legend()
ax_load.grid(alpha=0.4)
ax_load.set_title(
    'Per-gNB total useful PRB load  (paper KPM metric)\n'
    '3 UEs start at gNB-0 → transferred one by one'
)

ax_std.plot(df_ho.time_s, df_ho['std'], color='purple', lw=2,
            label='StdDev (paper reward target)')
for t_s, *_ in transfer_log:
    ax_std.axvline(t_s, color='crimson', lw=2, linestyle='--', alpha=0.8)
ax_std.set_ylabel('StdDev')
ax_std.set_xlabel('time (s)')
ax_std.legend(fontsize=9)
ax_std.grid(alpha=0.4)

plt.tight_layout()
plt.show()